# Getting Started with Semantic Kernel

This notebook demonstrates how to set up and use the Microsoft Semantic Kernel in .NET.

**Step 1**: Install NuGet packages

To get started with Semantic Kernel, you need to install the required NuGet packages. These packages provide the core functionality for interacting with AI models and managing environment variables. Specifically:
- `Microsoft.SemanticKernel` enables you to build and run AI-powered workflows.
- `DotNetEnv` allows you to load environment variables from a `.env` file, making it easier to manage secrets and configuration settings.

In [3]:
// Import Semantic Kernel
#r "nuget: Microsoft.SemanticKernel, 1.23.0"
#r "nuget: DotNetEnv, 3.1.0"

Installed Packages DotNetEnv, 3.1.0 Microsoft.SemanticKernel, 1.23.0

**Step 2**: Read environment variables

  In this step, we load these variables from a `.env` file (if present) so that they can be accessed by the application.


In [4]:
using DotNetEnv;
using System.IO;

var envFilePath = Path.Combine(Environment.CurrentDirectory, "../..", ".env");
if (File.Exists(envFilePath))
{
    Env.Load(envFilePath);
    Console.WriteLine($"Loaded environment variables from {envFilePath}");
}
else
{
    Console.WriteLine($"No .env file found at {envFilePath}");
}

Loaded environment variables from d:\personal\aiagent-workshop\notebooks\semantic-kernel-basic\../..\.env


**Step 3**: Instantiate the Kernel

The Semantic Kernel is the core component that orchestrates AI services and plugins. In this step, we create and configure a Kernel instance, which will be used to interact with AI models.

In [5]:
using System.ClientModel;
using OpenAI;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using System.Text;

OpenAIClient client = null;
if(Environment.GetEnvironmentVariable("USE_AZURE_OPENAI") == "true")
{
    // Configure Azure OpenAI client
    var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(azureEndpoint) });
}
else if(Environment.GetEnvironmentVariable("USE_OPENAI") == "true")
{
    // Configure OpenAI client
    var apiKey = Environment.GetEnvironmentVariable("OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey));
}
else if(Environment.GetEnvironmentVariable("USE_GITHUB") == "true")
{
    // Configure GitHub model client
    var uri = Environment.GetEnvironmentVariable("GITHUB_MODEL_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("GITHUB_TOKEN");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(uri) });
}

var modelId = Environment.GetEnvironmentVariable("MODEL");
// Create a chat completion service
var builder = Kernel.CreateBuilder();
builder.AddOpenAIChatCompletion(modelId, client);

// Get the chat completion service
Kernel kernel = builder.Build();
var chat = kernel.GetRequiredService<IChatCompletionService>(); 

Let's create a semantic function used to summarize content:

In [6]:
string skPrompt = """
{{$input}}

Summarize the content above.
""";

Let's configure the prompt settings

Using `FunctionChoiceBehavior.Auto()` will enable automatic function calling. There are also other options like `Required` or `None` which allow to control function calling behavior. More information about it can be found [here](https://learn.microsoft.com/en-gb/semantic-kernel/concepts/ai-services/chat-completion/function-calling/function-choice-behaviors?pivots=programming-language-csharp).

In [9]:
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Connectors.OpenAI;

#pragma warning disable SKEXP0001
var executionSettings = new OpenAIPromptExecutionSettings 
{
    FunctionChoiceBehavior = FunctionChoiceBehavior.Auto()
};
#pragma warning restore SKEXP0001


Let's define a new function that returns random weather information. We'll then call it using Semantic Kernel's function calling capabilities.

In [10]:
// Define a function that returns random weather
using System;
using Microsoft.SemanticKernel;

public class WeatherPlugin
{
    [KernelFunction]
    public string GetRandomWeather()
    {
        var weatherOptions = new[] { "Sunny", "Cloudy", "Rainy", "Windy", "Stormy", "Snowy" };
        var rnd = new Random();
        var temp = rnd.Next(10, 35);
        var weather = weatherOptions[rnd.Next(weatherOptions.Length)];
        return $"The weather is {weather} and {temp}°C.";
    }
}

Now let's register this function with the kernel and call it using function calling.

In [11]:
// Register the WeatherPlugin with the kernel
var plugin = new WeatherPlugin();
kernel.ImportPluginFromObject(plugin, "weather");

// Create a prompt that will trigger the function call
string weatherPrompt = "What is the weather today?";

// Use the execution settings for function calling
var result = await chat.GetChatMessageContentAsync(weatherPrompt, executionSettings, kernel);
Console.WriteLine($"AI: {result.Content}");

AI: The weather today is sunny with a temperature of 22°C.
